# Avito text orientation — воспроизведение финального решения

Задача — оценить вероятность того, что изображение текста перевёрнуто на 180°. OCR не является целью. Финальный результат получен ансамблем **75% robust ViT-B/16 + 25% robust MobileNetV3-Large**.

Модели обучались на детерминированных синтетических парах: сначала создаётся прямое изображение, затем вторым элементом пары служит его точный поворот на 180°. Robust-профиль добавляет только в train умеренную перспективу, crop, blur/downscale, тени/блики и occlusion; validation остаётся неизменной. На inference модель получает исходное и повёрнутое изображения, вероятности симметризуются и калибруются temperature scaling, выбранным по 5-fold OOF validation.

Зафиксированы seed, порядок изображений, хэши checkpoint-файлов и протокол validation. Компоненты выполняются последовательно, поэтому notebook помещается в память обычного Colab GPU.

## Итоговые результаты

| Решение | Validation 1 − Brier | Hidden test 1 − Brier |
|---|---:|---:|
| ViT-B/16 | 0.943337 | 0.905859 |
| Robust ViT-B/16 | 0.954320 | 0.928290 |
| 25% robust MobileNet + 75% robust ViT | **0.956555** | **0.929383** |

Отправленный файл содержит 20 000 строк и имеет SHA-256 `cf3be491f4df56015bce7f78f5db9c35ea7ad5f651d6fdf801ebff99fd1a5482`.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
REVISION = "solution-v2"
ARTIFACT_URL = "https://github.com/frest1ler/text-orientation-classification/releases/download/solution-v1/text-orientation-solution-artifacts.zip"
ARTIFACT_SHA256 = "08864ad16e7df88489c6479a060c817d9d9041a6c75674f077a7167954583502"
TEST_ZIP_PATH = "/content/test.zip"  # можно заменить путём к выданному test.zip на Google Drive
PROJECT_DIR = "/content/text-orientation-solution"
BATCH_SIZE = None  # None: MobileNet=128, ViT=32
NUM_WORKERS = 2
RUN_TESTS = False

## 1. Код и зависимости

Клонируется открытый репозиторий. Вся содержательная логика находится в тестируемых модулях `src/` и `scripts/`; notebook остаётся короткой воспроизводимой точкой входа.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

repo = Path("/content/text-orientation-classification")
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "checkout", REVISION], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 2. Данные и зафиксированные веса

Если `/content/test.zip` отсутствует, появится стандартное окно загрузки Colab. Bundle содержит только веса, калибраторы и manifest ансамбля — тестовые изображения в него не входят. Перед распаковкой проверяется SHA-256.

In [ ]:
test_zip = Path(TEST_ZIP_PATH)
if not test_zip.is_file():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Загрузите ровно один выданный test.zip")
    source_name = next(iter(uploaded))
    test_zip = Path("/content") / source_name

bundle = Path("/content/text-orientation-solution-artifacts.zip")
if not bundle.is_file():
    import gdown
    gdown.download(ARTIFACT_URL, str(bundle), quiet=False, fuzzy=True)
digest_builder = hashlib.sha256()
with bundle.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest_builder.update(chunk)
digest = digest_builder.hexdigest()
if digest != ARTIFACT_SHA256:
    raise ValueError(f"Artifact SHA-256 mismatch: {digest}")

project = Path(PROJECT_DIR)
(project / "data").mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(project)
destination = project / "data/test.zip"
if destination.resolve() != test_zip.resolve():
    shutil.copy2(test_zip, destination)
print({"test_zip": str(destination), "artifact_sha256": digest})

## 3. Проверка manifest и полный inference

Registry проверяет SHA-256 обоих checkpoint-файлов и калибраторов. Затем модели выполняются последовательно, а итоговые вероятности смешиваются весами из зафиксированного manifest. Recovery сохраняет промежуточные предсказания и позволяет продолжить оборванный запуск.

In [ ]:
ensemble_leaderboard = project / "registry/ensembles/leaderboard.json"
if not ensemble_leaderboard.is_file():
    raise FileNotFoundError(f"Missing ensemble registry: {ensemble_leaderboard}")
print(ensemble_leaderboard.read_text(encoding="utf-8"))
command = [
    sys.executable, "-m", "scripts.infer_ensemble",
    "--project-dir", str(project),
    "--name", "best_ensemble",
    "--num-workers", str(NUM_WORKERS),
    "--contact-sheet-count", "16",
]
if BATCH_SIZE is not None:
    command += ["--batch-size", str(BATCH_SIZE)]
subprocess.run(command, check=True)

## 4. Проверка и скачивание submission.csv

Финальная проверка требует точные колонки шаблона, 20 000 уникальных ID и вероятности в `[0, 1]`. SHA-256 может отличаться только если проверяющий использует другой архив тестовых данных или другую платформенную реализацию чисел с плавающей точкой.

In [ ]:
import csv
outputs = sorted((project / "inference/runs").glob("best_ensemble_full_*/submission.csv"))
if not outputs:
    raise FileNotFoundError("Полный submission.csv не создан")
submission = outputs[-1]
with submission.open(encoding="utf-8", newline="") as stream:
    reader = csv.DictReader(stream)
    rows = list(reader)
if reader.fieldnames != ["image_id", "p_180"]:
    raise ValueError(f"Unexpected columns: {reader.fieldnames}")
if len(rows) != 20_000 or len({row["image_id"] for row in rows}) != 20_000:
    raise ValueError("Submission must contain 20,000 unique image IDs")
probabilities = [float(row["p_180"]) for row in rows]
if not all(0.0 <= value <= 1.0 for value in probabilities):
    raise ValueError("Probabilities must be in [0, 1]")
submission_sha256 = hashlib.sha256(submission.read_bytes()).hexdigest()
print({"submission": str(submission), "rows": len(rows), "sha256": submission_sha256})
try:
    from google.colab import files
    files.download(str(submission))
except ImportError:
    pass